In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
)

# -------------------------------------------------------------------
# STEP 1: INPUT - Load and understand the dataset
# -------------------------------------------------------------------
iris = load_iris()
X = iris.data          # shape (150, 4) -> sepal length/width, petal length/width
y = iris.target        # shape (150,)   -> 0=setosa, 1=versicolor, 2=virginica
class_names = iris.target_names

print("=" * 60)
print("STEP 1: DATASET OVERVIEW")
print("=" * 60)
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]} -> {iris.feature_names}")
print(f"Classes: {len(class_names)} -> {list(class_names)}")
print(f"Class balance: {np.bincount(y)} (50 each, perfectly balanced)\n")

# -------------------------------------------------------------------
# STEP 2: PROCESS - Split data into training and testing sets
# -------------------------------------------------------------------
# 80/20 split as shown in "The Full Architecture" slide.
# stratify=y keeps the 3 classes balanced in both train and test sets.
# random_state fixes the shuffle so results are reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 60)
print("STEP 2: TRAIN-TEST SPLIT")
print("=" * 60)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}\n")

# -------------------------------------------------------------------
# STEP 3: PROCESS - Feature Scaling ("The Gatekeeper Rule")
# -------------------------------------------------------------------
# KNN is distance-based, so features must be on the same scale.
# IMPORTANT: fit the scaler ONLY on training data, then transform both,
# to avoid leaking information from the test set.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=" * 60)
print("STEP 3: FEATURE SCALING (StandardScaler: mean=0, variance=1)")
print("=" * 60)
print(f"Before scaling -> mean: {X_train[:, 0].mean():.2f}, std: {X_train[:, 0].std():.2f}")
print(f"After scaling  -> mean: {X_train_scaled[:, 0].mean():.2f}, std: {X_train_scaled[:, 0].std():.2f}\n")

# -------------------------------------------------------------------
# STEP 4: PROCESS - Apply the KNN classification algorithm
# -------------------------------------------------------------------
# Scikit-learn workflow: Instantiate -> Fit -> Predict
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("=" * 60)
print("STEP 4: MODEL TRAINING (KNN, k=5)")
print("=" * 60)
print("Model trained successfully.\n")

# -------------------------------------------------------------------
# STEP 5: OUTPUT - Validate with Confusion Matrix and F1 Score
# -------------------------------------------------------------------
cm = confusion_matrix(y_test, predictions)
acc = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average="macro")

print("=" * 60)
print("STEP 5: OUTPUT VALIDATION")
print("=" * 60)
print("Confusion Matrix (rows=actual, cols=predicted):")
print(f"{'':>12}{'setosa':>10}{'versicolor':>12}{'virginica':>11}")
for i, row in enumerate(cm):
    print(f"{class_names[i]:>12}{row[0]:>10}{row[1]:>12}{row[2]:>11}")

print(f"\nAccuracy:  {acc:.2%}")
print(f"F1 Score (macro avg): {f1:.4f}\n")

print("Full Classification Report:")
print(classification_report(y_test, predictions, target_names=class_names))

# -------------------------------------------------------------------
# STEP 6: BONUS - Find the optimal K ("Tuning the Engine")
# -------------------------------------------------------------------
print("=" * 60)
print("BONUS: CHOOSING THE OPTIMAL K (Elbow Method)")
print("=" * 60)
error_rates = []
k_range = range(1, 21)
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pred_k = knn.predict(X_test_scaled)
    error_rates.append(np.mean(pred_k != y_test))

best_k = list(k_range)[int(np.argmin(error_rates))]
print(f"Best K found: {best_k} (lowest error rate: {min(error_rates):.4f})")

STEP 1: DATASET OVERVIEW
Samples: 150
Features: 4 -> ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: 3 -> [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Class balance: [50 50 50] (50 each, perfectly balanced)

STEP 2: TRAIN-TEST SPLIT
Training samples: 120
Testing samples:  30

STEP 3: FEATURE SCALING (StandardScaler: mean=0, variance=1)
Before scaling -> mean: 5.84, std: 0.84
After scaling  -> mean: -0.00, std: 1.00

STEP 4: MODEL TRAINING (KNN, k=5)
Model trained successfully.

STEP 5: OUTPUT VALIDATION
Confusion Matrix (rows=actual, cols=predicted):
                setosa  versicolor  virginica
      setosa        10           0          0
  versicolor         0          10          0
   virginica         0           2          8

Accuracy:  93.33%
F1 Score (macro avg): 0.9327

Full Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versico